# Mushroom Risk Comparison Notebook

This notebook is the clean reproducible notebook for the project. The core task is 4-class mushroom risk classification using one baseline and three later comparison models.

## Research Question

**Which fine-tuned architecture performs best on mushroom risk classification, and which model most reduces dangerous mistakes where unsafe mushrooms are predicted as safe?**

## Safety Note

This is a research prototype only. It must not be used to decide whether a mushroom is safe to eat.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.mushroom_risk import ExperimentConfig, run_experiment


## Mode

Leave `RUN_TRAINING = False` to analyze an existing finished run. Set it to `True` only if you want the notebook to launch training.

In [ ]:
RUN_TRAINING = False
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "mushroom_comparison"

config = ExperimentConfig(
    epochs=12,
    batch_size=8,
    num_workers=0,
    output_dir=str(OUTPUT_DIR),
)

if RUN_TRAINING:
    run_experiment(config)

metadata = json.loads((OUTPUT_DIR / "metadata.json").read_text())
results = pd.read_csv(OUTPUT_DIR / "results.csv").sort_values("test_f1_macro", ascending=False)
results


In [ ]:
summary = {
    "dataset_root": metadata["dataset_root"],
    "split_sizes": metadata["split_sizes"],
    "class_distribution": metadata["class_distribution"],
    "class_weights": metadata["class_weights"],
}
summary


In [ ]:
display(Image(filename=str(OUTPUT_DIR / "model_comparison.png")))


In [ ]:
for model_name in results["model_name"]:
    print(f"\n=== {model_name} ===")
    display(pd.read_csv(OUTPUT_DIR / f"{model_name}_per_class_metrics.csv"))
    display(Image(filename=str(OUTPUT_DIR / f"{model_name}_confusion.png")))
